In [ ]:
import os
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
import warnings
warnings.filterwarnings("ignore")
import random
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import Dataset
from transformers import (
    AutoConfig,
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding
)

# 変数、乱数の設定

In [ ]:
class CFG:
    VER = 1
    AUTHOR = "suba"
    COMPETITION = "atmacup17"
    DATA_PATH = Path("/workspace/kaggle_llm_book/ch3/data/takaito/atmacup17")
    # MODEL_PATH = "/kaggle/input/kaggle-llm-book-3-4-bert-2025-09/suba_deberta-v3-large_seed42_ver1-fold0/checkpoint-80"
    MODEL_NAME = "deberta-v3-large"
    MAX_LENGTH = 2048
    EPOCHS = 3
    STEPS = 20
    WARMUP_RATIO = 0.05
    TRAIN_BATCH_SPLIT = 8
    TRAIN_BATCH_SIZE = 64
    EVAL_BATCH_SIZE = 64
    LEARNING_RATE = 2e-5
    OPTIM = "adamw_torch"
    USE_GPU = torch.cuda.is_available()
    SEED = 42
    N_SPLIT = 2
    TARGET_COL = "Recommended IND"
    TARGET_COL_CLASS_NUM = 2
    METRIC = "auc"
    METRIC_MAXIMIZE_FLAG = True
    PREFIX  = f"{AUTHOR}_{MODEL_NAME}_seed{SEED}_ver{VER}"
    MODEL_PATH_DICT = {
        0: "/kaggle/input/kaggle-llm-book-3-4-bert-2025-09/suba_deberta-v3-large_seed42_ver1-fold0/checkpoint-80",
        1: "/kaggle/input/kaggle-llm-book-3-4-bert-2025-09/suba_deberta-v3-large_seed42_ver1-fold1/checkpoint-100"
    }

In [ ]:
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    if CFG.USE_GPU:
        torch.cuda.manual_seed(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
seed_everything(CFG.SEED)

# データの読み込み、前処理

In [ ]:
clothing_master_df = pd.read_csv(CFG.DATA_PATH / "clothing_master.csv")
test_df = pd.read_csv(CFG.DATA_PATH / "test.csv")

In [ ]:
def make_prompt_column(df):
    df["prompt"] = df["Title"] + " " + df["Review Text"]
    return df

def preprocessing(df, clothing_master_df):
    df["Title"] = df["Title"].fillna("")
    df["Review Text"] = df["Review Text"].fillna("")
    df = df.merge(clothing_master_df, on="Clothing ID", how="left")
    df = make_prompt_column(df)
    return df

In [ ]:
test_df = preprocessing(test_df, clothing_master_df)

In [ ]:
display(test_df)

In [ ]:
print(test_df[["prompt"]].iloc[0].values)

# トークナイザの設定

# データセットの作成

# 推論